# 01 · Bronze layer: raw ingestion

**Purpose.** Load the car insurance claim dataset used in Phases 1 and 2 and keep an unmodified raw copy as a Delta table. Nothing is cleaned here.

**Inputs** (files in the Unity Catalog volume `/Volumes/workspace/car_insurance/raw/`)

| File | What it is |
|---|---|
| `Car_Insurance_Claim.csv` | Source dataset: 10,000 drivers, 19 columns, CSV with a header row, about 1.3 MB |
| `split_assignment.csv` | `ID → train / validation / test` lookup that reproduces the Phase 2 scikit-learn split (written by `src/make_split.py`) |

**Outputs** (Delta tables in `workspace.car_insurance`)

| Table | Contents |
|---|---|
| `claims_bronze` | Every source row and column as raw strings, plus `_corrupt_record`, `_source_file`, `_ingested_at` |
| `claims_split_bronze` | The split lookup as raw strings, plus `_source_file`, `_ingested_at` |

**Assumptions**

- The catalog is named `workspace` (the Free Edition default). Change `CATALOG` in the configuration cell otherwise.
- The source file has one record per line (no quoted line breaks), so its line count minus the header is the expected record count.
- Runs on serverless compute. The notebook can be rerun from top to bottom: every table is overwritten.

**How to run.** Run all cells. On a brand-new workspace the first run creates the schema and volume and then stops, asking for the two files. Upload them to the volume (Catalog → `workspace` → `car_insurance` → `raw` → *Upload to this volume*) and run all again.

**Dataset source.** Kaggle, *Car Insurance Data* (`Car_Insurance_Claim.csv`). The same file was read from the team GitHub repository in Phases 1 and 2.

In [ ]:
# --- Configuration (identical in every Phase 3 notebook) ----------------------
# Free Edition workspaces come with a Unity Catalog catalog named `workspace`.
# If yours is named differently, change CATALOG here and in the other notebooks.
CATALOG = "workspace"
SCHEMA = "car_insurance"
VOLUME = "raw"

RAW_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CLAIMS_CSV = f"{RAW_DIR}/Car_Insurance_Claim.csv"
SPLIT_CSV = f"{RAW_DIR}/split_assignment.csv"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_bronze"
SPLIT_BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_split_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.claims_silver"
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_features"
GOLD_SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_summary"

# The 19 columns of the source file, in file order.
SOURCE_COLUMNS = [
    "ID", "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
    "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN",
    "POSTAL_CODE", "ANNUAL_MILEAGE", "VEHICLE_TYPE", "SPEEDING_VIOLATIONS",
    "DUIS", "PAST_ACCIDENTS", "OUTCOME",
]

In [ ]:
from pyspark.sql import functions as F


def write_delta(df, table_name, comment):
    """Overwrite a managed Delta table so the notebook can be rerun top to bottom."""
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    spark.sql(f"COMMENT ON TABLE {table_name} IS '{comment}'")


def validate_table(table_name, expected_rows, expected_columns):
    """Read a table back from the catalog and compare it with what was written."""
    saved = spark.table(table_name)
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
    rows, columns = saved.count(), len(saved.columns)
    passed = (detail["format"] == "delta" and rows == expected_rows
              and columns == expected_columns)
    return (table_name, detail["format"], expected_rows, rows,
            expected_columns, columns, "PASS" if passed else "FAIL")


VALIDATION_SCHEMA = ("table string, format string, expected_rows long, actual_rows long, "
                     "expected_columns int, actual_columns int, status string")


def show_validation(results):
    display(spark.createDataFrame(results, VALIDATION_SCHEMA))
    failed = [r[0] for r in results if r[-1] != "PASS"]
    assert not failed, f"Validation failed for: {failed}"

## 1. Create the schema and volume, and check the source files

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")


def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


missing = [path for path in (CLAIMS_CSV, SPLIT_CSV) if not file_exists(path)]
if missing:
    raise FileNotFoundError(
        "Upload these files to the volume, then run the notebook again:\n  "
        + "\n  ".join(missing))
print("Source files found in", RAW_DIR)

## 2. Read the source file with Spark

Every column is read as a **string**. Bronze should hold the values exactly as they appear in the file (for example `OUTCOME` is written as `0.0` / `1.0`), and reading strings means no value can be changed or nulled by type inference. Types are assigned in the Silver notebook, where each cast is checked.

Malformed records are **not dropped**. The reader runs in `PERMISSIVE` mode with a `_corrupt_record` column: a line that does not match the 19-column layout is kept, with its original text stored in that column, so it can be counted here and handled explicitly in Silver.

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType

# The header must match the layout this pipeline was written for.
header_columns = spark.read.option("header", "true").csv(CLAIMS_CSV).columns
assert header_columns == SOURCE_COLUMNS, f"Unexpected header: {header_columns}"

# Independent record count taken from the raw text, before any parsing.
raw_line_count = spark.read.text(CLAIMS_CSV).count()
expected_records = raw_line_count - 1  # minus the header line
print(f"Lines in file: {raw_line_count:,}  ->  expected records: {expected_records:,}")

bronze_schema = StructType(
    [StructField(name, StringType(), True) for name in SOURCE_COLUMNS]
    + [StructField("_corrupt_record", StringType(), True)]
)

raw_df = (
    spark.read.format("csv")
    .schema(bronze_schema)
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(CLAIMS_CSV)
)

# Technical ingestion metadata. Columns starting with "_" are not source fields.
bronze_df = raw_df.select(
    "*",
    F.col("_metadata.file_path").alias("_source_file"),
    F.current_timestamp().alias("_ingested_at"),
)

bronze_df.printSchema()

In [ ]:
display(bronze_df.limit(10))

## 3. Write the Bronze Delta table

In [ ]:
write_delta(
    bronze_df,
    BRONZE_TABLE,
    "Raw car insurance claim records, all source columns as strings. One row per source line.",
)
print("Wrote", BRONZE_TABLE)

## 4. Baseline checks on the saved table

These checks read the table back from the catalog. They record the baseline row count and show whether any record failed to parse.

In [ ]:
bronze = spark.table(BRONZE_TABLE)

baseline_rows = bronze.count()
malformed_rows = bronze.filter(F.col("_corrupt_record").isNotNull()).count()

print(f"Records expected from the raw file : {expected_records:,}")
print(f"Rows in {BRONZE_TABLE} : {baseline_rows:,}")
print(f"Malformed records kept (flagged)   : {malformed_rows:,}")

assert baseline_rows == expected_records, "Row count differs from the raw file"

In [ ]:
# Missing values per source column. In Bronze these are only counted, not fixed.
null_counts = bronze.select(
    [F.sum(F.col(name).isNull().cast("int")).alias(name) for name in SOURCE_COLUMNS]
).first().asDict()

display(
    spark.createDataFrame(
        [(name, int(count), round(100 * count / baseline_rows, 2))
         for name, count in null_counts.items()],
        "column string, missing_values long, missing_pct double",
    )
)

## 5. Ingest the Phase 2 split lookup

Phase 2 split the drivers with scikit-learn (`train_test_split`, `random_state=123`). Spark cannot reproduce that shuffle, so `src/make_split.py` reruns the two scikit-learn calls locally and saves which split each `ID` belongs to. Loading that lookup lets the Spark models train and test on exactly the same drivers as Phase 2.

The lookup contains only `ID` and the split name. It carries no feature or target values.

In [ ]:
split_header = spark.read.option("header", "true").csv(SPLIT_CSV).columns
assert split_header == ["ID", "split"], f"Unexpected header: {split_header}"
expected_split_records = spark.read.text(SPLIT_CSV).count() - 1  # minus the header line

split_df = (
    spark.read.format("csv")
    .schema("ID string, split string")
    .option("header", "true")
    .load(SPLIT_CSV)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.current_timestamp().alias("_ingested_at"),
    )
)

write_delta(
    split_df,
    SPLIT_BRONZE_TABLE,
    "Phase 2 train/validation/test assignment per driver ID, as raw strings.",
)

display(spark.table(SPLIT_BRONZE_TABLE).groupBy("split").count().orderBy("split"))

## 6. Delta validation: read back and verify

Each table is read back from the catalog and checked for Delta format, row count and column count.

In [ ]:
show_validation([
    validate_table(BRONZE_TABLE, expected_records, len(SOURCE_COLUMNS) + 3),
    validate_table(SPLIT_BRONZE_TABLE, expected_split_records, 4),
])

**Result.** The Bronze table holds one row per line of the source file, with every source value preserved as text. No record was malformed. The only missing values are in `CREDIT_SCORE` and `ANNUAL_MILEAGE`; they are left as they are and handled downstream.

Next: `02_silver_layer`.